# Optuna Hyperparameter Tuning

Bayesian optimization of LightGBM hyperparameters for all 3 evaluation levels.

Uses the best feature set: summary-stats (288) + survey (27) = 315 features.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from config import (RANDOM_STATE, N_FOLDS, LABEL_MAP_4_TO_3, LABEL_MAP_4_TO_2,
                    CLASS_NAMES_4, CLASS_NAMES_3, CLASS_NAMES_2, set_seed)

optuna.logging.set_verbosity(optuna.logging.WARNING)
FEATURES_DIR = Path('../outputs/features')
set_seed()
print('Ready')

## Load Data

In [ ]:
X_orig = np.load(FEATURES_DIR / 'X.npy')
X_c    = np.load(FEATURES_DIR / 'X_option_c.npy')
X_survey = np.load(FEATURES_DIR / 'X_survey.npy')
y      = np.load(FEATURES_DIR / 'y.npy')
lengths = np.load(FEATURES_DIR / 'lengths.npy')

X_wear = np.concatenate([X_orig, X_c], axis=2)
N, T, F = X_wear.shape

def flatten_temporal(X_3d, lengths):
    N, T, F = X_3d.shape
    stats = []
    for f in range(F):
        col = X_3d[:, :, f]
        col_mean = np.nanmean(col, axis=1)
        col_std  = np.nanstd(col, axis=1)
        col_min  = np.nanmin(col, axis=1)
        col_max  = np.nanmax(col, axis=1)
        col_range = col_max - col_min
        slope = np.zeros(N)
        diff = np.zeros(N)
        for i in range(N):
            valid = col[i, :int(lengths[i])]
            valid = valid[~np.isnan(valid)]
            if len(valid) >= 2:
                slope[i] = np.polyfit(np.arange(len(valid)), valid, 1)[0]
                diff[i] = valid[-1] - valid[0]
        stats.extend([col_mean, col_std, col_min, col_max, col_range, slope, diff])
    stats.append(lengths.astype(float))
    return np.column_stack(stats)

import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    X_flat = flatten_temporal(X_wear, lengths)

X_all = np.concatenate([X_flat, X_survey], axis=1)
print(f'Features: {X_all.shape[1]}  Samples: {N}')

X_perday = X_wear.reshape(N, T * F)
X_perday_all = np.concatenate([X_perday, X_flat, X_survey], axis=1)
print(f'Per-day+agg features: {X_perday_all.shape[1]}')

## Optuna — 4-Class AUC

In [ ]:
def objective_4class(trial):
    params = {
        'objective': 'multiclass', 'num_class': 4, 'metric': 'multi_logloss',
        'boosting_type': 'gbdt', 'verbose': -1, 'is_unbalance': True,
        'random_state': RANDOM_STATE,
        'n_estimators': 2000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 60),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 5.0),
    }
    
    feat_set = trial.suggest_categorical('feat_set', ['summary', 'perday'])
    X = X_all if feat_set == 'summary' else X_perday_all
    
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros((N, 4))
    for tr, va in skf.split(X, y):
        model = lgb.LGBMClassifier(**params)
        model.fit(X[tr], y[tr], eval_set=[(X[va], y[va])],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        oof[va] = model.predict_proba(X[va])
    return roc_auc_score(y, oof, multi_class='ovr', average='macro')

study_4 = optuna.create_study(direction='maximize', study_name='lgb_4class')
study_4.optimize(objective_4class, n_trials=300, show_progress_bar=True)

print(f'\nBest 4-AUC: {study_4.best_value:.4f}')
print(f'Best params: {study_4.best_params}')

## Optuna — 3-Class AUC

In [ ]:
y3 = np.array([LABEL_MAP_4_TO_3[i] for i in y])

def objective_3class(trial):
    params = {
        'objective': 'multiclass', 'num_class': 3, 'metric': 'multi_logloss',
        'boosting_type': 'gbdt', 'verbose': -1, 'is_unbalance': True,
        'random_state': RANDOM_STATE,
        'n_estimators': 2000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 60),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 5.0),
    }
    feat_set = trial.suggest_categorical('feat_set', ['summary', 'perday'])
    X = X_all if feat_set == 'summary' else X_perday_all
    
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros((N, 3))
    for tr, va in skf.split(X, y3):
        model = lgb.LGBMClassifier(**params)
        model.fit(X[tr], y3[tr], eval_set=[(X[va], y3[va])],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        oof[va] = model.predict_proba(X[va])
    return roc_auc_score(y3, oof, multi_class='ovr', average='macro')

study_3 = optuna.create_study(direction='maximize', study_name='lgb_3class')
study_3.optimize(objective_3class, n_trials=300, show_progress_bar=True)

print(f'\nBest 3-AUC: {study_3.best_value:.4f}')
print(f'Best params: {study_3.best_params}')

## Optuna — 2-Class AUC

In [ ]:
y2 = np.array([LABEL_MAP_4_TO_2[i] for i in y])

def objective_2class(trial):
    params = {
        'objective': 'binary', 'metric': 'binary_logloss',
        'boosting_type': 'gbdt', 'verbose': -1, 'is_unbalance': True,
        'random_state': RANDOM_STATE,
        'n_estimators': 2000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 60),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 5.0),
    }
    feat_set = trial.suggest_categorical('feat_set', ['summary', 'perday'])
    X = X_all if feat_set == 'summary' else X_perday_all
    
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(N)
    for tr, va in skf.split(X, y2):
        model = lgb.LGBMClassifier(**params)
        model.fit(X[tr], y2[tr], eval_set=[(X[va], y2[va])],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        oof[va] = model.predict_proba(X[va])[:, 1]
    return roc_auc_score(y2, oof)

study_2 = optuna.create_study(direction='maximize', study_name='lgb_2class')
study_2.optimize(objective_2class, n_trials=300, show_progress_bar=True)

print(f'\nBest 2-AUC: {study_2.best_value:.4f}')
print(f'Best params: {study_2.best_params}')

## Retrain Best Models + Get OOF + Ensemble

In [ ]:
def compute_all_metrics(y4, proba4, label=''):
    pred = np.argmax(proba4, axis=1)
    acc = accuracy_score(y4, pred)
    auc4 = roc_auc_score(y4, proba4, multi_class='ovr', average='macro')
    y3 = np.array([LABEL_MAP_4_TO_3[i] for i in y4])
    p3 = np.zeros((len(y4), 3))
    p3[:, 0] = proba4[:, 0]; p3[:, 1] = proba4[:, 1]
    p3[:, 2] = proba4[:, 2] + proba4[:, 3]
    auc3 = roc_auc_score(y3, p3, multi_class='ovr', average='macro')
    y2 = np.array([LABEL_MAP_4_TO_2[i] for i in y4])
    auc2 = roc_auc_score(y2, proba4[:, 1] + proba4[:, 2] + proba4[:, 3])
    print(f'\n  {label}')
    print(f'  acc={acc*100:.1f}%  4-AUC={auc4:.4f}  3-AUC={auc3:.4f}  2-AUC={auc2:.4f}')
    print(classification_report(y4, pred, target_names=CLASS_NAMES_4, digits=3))
    return acc, auc4, auc3, auc2

# Retrain 4-class with best Optuna params
bp4 = study_4.best_params.copy()
feat4 = bp4.pop('feat_set')
X4 = X_all if feat4 == 'summary' else X_perday_all
bp4.update({'objective': 'multiclass', 'num_class': 4, 'metric': 'multi_logloss',
            'boosting_type': 'gbdt', 'verbose': -1, 'is_unbalance': True,
            'random_state': RANDOM_STATE, 'n_estimators': 2000})

set_seed()
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
oof_opt4 = np.zeros((N, 4))
for fold, (tr, va) in enumerate(skf.split(X4, y)):
    model = lgb.LGBMClassifier(**bp4)
    model.fit(X4[tr], y[tr], eval_set=[(X4[va], y[va])],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    oof_opt4[va] = model.predict_proba(X4[va])
    print(f'  4-class fold {fold+1}: {model.best_iteration_} rounds')

np.save(FEATURES_DIR / 'oof_optuna_4class.npy', oof_opt4)
acc_o4, auc4_o4, auc3_o4, auc2_o4 = compute_all_metrics(y, oof_opt4, 'Optuna LGB 4-class')

In [ ]:
# Retrain 3-class
bp3 = study_3.best_params.copy()
feat3 = bp3.pop('feat_set')
X3 = X_all if feat3 == 'summary' else X_perday_all
bp3.update({'objective': 'multiclass', 'num_class': 3, 'metric': 'multi_logloss',
            'boosting_type': 'gbdt', 'verbose': -1, 'is_unbalance': True,
            'random_state': RANDOM_STATE, 'n_estimators': 2000})

set_seed()
oof_opt3 = np.zeros((N, 3))
for fold, (tr, va) in enumerate(skf.split(X3, y3)):
    model = lgb.LGBMClassifier(**bp3)
    model.fit(X3[tr], y3[tr], eval_set=[(X3[va], y3[va])],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    oof_opt3[va] = model.predict_proba(X3[va])
    print(f'  3-class fold {fold+1}: {model.best_iteration_} rounds')

np.save(FEATURES_DIR / 'oof_optuna_3class.npy', oof_opt3)
auc3_opt = roc_auc_score(y3, oof_opt3, multi_class='ovr', average='macro')
print(f'\n  Optuna LGB 3-class: 3-AUC={auc3_opt:.4f}')

In [ ]:
# Retrain 2-class
bp2 = study_2.best_params.copy()
feat2 = bp2.pop('feat_set')
X2 = X_all if feat2 == 'summary' else X_perday_all
bp2.update({'objective': 'binary', 'metric': 'binary_logloss',
            'boosting_type': 'gbdt', 'verbose': -1, 'is_unbalance': True,
            'random_state': RANDOM_STATE, 'n_estimators': 2000})

set_seed()
oof_opt2 = np.zeros(N)
for fold, (tr, va) in enumerate(skf.split(X2, y2)):
    model = lgb.LGBMClassifier(**bp2)
    model.fit(X2[tr], y2[tr], eval_set=[(X2[va], y2[va])],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    oof_opt2[va] = model.predict_proba(X2[va])[:, 1]
    print(f'  2-class fold {fold+1}: {model.best_iteration_} rounds')

np.save(FEATURES_DIR / 'oof_optuna_2class.npy', oof_opt2)
auc2_opt = roc_auc_score(y2, oof_opt2)
print(f'\n  Optuna LGB 2-class: 2-AUC={auc2_opt:.4f}')

## Ensemble: Optuna LGB + Hybrid LSTM

In [ ]:
hybrid_oof = np.load(FEATURES_DIR / 'oof_hybrid_T2_a0.5.npy')

print('=== Optuna LGB + Hybrid LSTM Ensembles ===')
best_ens_auc4 = 0
best_ens_w = 0
for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    ens = w * oof_opt4 + (1 - w) * hybrid_oof
    _, a4, a3, a2 = compute_all_metrics(y, ens, f'OptLGB×{w:.1f} + Hybrid×{1-w:.1f}')
    if a4 > best_ens_auc4:
        best_ens_auc4 = a4
        best_ens_w = w
        best_ens_metrics = (a4, a3, a2)

## Final Summary

In [ ]:
print(f"{'Model':<55} {'4-AUC':>7} {'3-AUC':>7} {'2-AUC':>7}")
print('=' * 80)
print(f"{'BASELINE: LSTM distilled (wearable only)':<55} {'0.6879':>7} {'0.6858':>7} {'0.6846':>7}")
print(f"{'Hybrid LSTM (wearable+survey)':<55} {'0.7167':>7} {'0.7213':>7} {'0.7474':>7}")
print(f"{'LGB manual (summary+survey)':<55} {'0.7024':>7} {'0.7321':>7} {'0.7718':>7}")
print(f"{'Prev best ensemble':<55} {'0.7338':>7} {'0.7407':>7} {'0.7852':>7}")
print('-' * 80)
print(f"{'Optuna LGB 4-class':<55} {auc4_o4:>7.4f} {auc3_o4:>7.4f} {auc2_o4:>7.4f}")
print(f"{'Optuna LGB 3-class (dedicated)':<55} {'—':>7} {auc3_opt:>7.4f} {'—':>7}")
print(f"{'Optuna LGB 2-class (dedicated)':<55} {'—':>7} {'—':>7} {auc2_opt:>7.4f}")
print(f"{'Optuna LGB + Hybrid ensemble (best w={best_ens_w})':<55} {best_ens_metrics[0]:>7.4f} {best_ens_metrics[1]:>7.4f} {best_ens_metrics[2]:>7.4f}")
print('-' * 80)
print(f'\nTotal improvement from baseline (0.6879):')
print(f'  4-AUC: {best_ens_metrics[0] - 0.6879:+.4f}')
print(f'  3-AUC: {best_ens_metrics[1] - 0.6858:+.4f}')
print(f'  2-AUC: {best_ens_metrics[2] - 0.6846:+.4f}')
print(f'\nOptuna best params:')
print(f'  4-class: {study_4.best_params}')
print(f'  3-class: {study_3.best_params}')
print(f'  2-class: {study_2.best_params}')